# U2-05 · Regularización, redes convolucionales, recurrentes y autocodificador

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 2, apartados 2.6 a 2.9*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con el botón ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) todo va bastante más rápido; sin ella, las celdas 1, 3 y 4 tardan varios minutos cada una.

> Datos: Fashion-MNIST, de Zalando Research (licencia MIT), y MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges (CC BY-SA 3.0). Keras los descarga en la primera ejecución.

## 1. Regularización y matriz de confusión (2.6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x / 255.0, x_pru / 255.0
x_ent, y_ent = x[:50_000], y[:50_000]          # entrenamiento
x_val, y_val = x[50_000:], y[50_000:]          # validación

def red(extra=(), drop=0.0, l2=0.0):
    keras.utils.set_random_seed(0)
    reg = keras.regularizers.L2(l2) if l2 else None
    capas = [keras.Input(shape=(28, 28, 1)), *extra, layers.Flatten()]
    for n in (256, 128):
        capas += [layers.Dense(n, activation="relu",
                               kernel_regularizer=reg),
                  layers.Dropout(drop)]
    m = keras.Sequential(capas + [layers.Dense(10, "softmax")])
    m.compile("adam", "sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

aumento = (layers.RandomFlip("horizontal"),       # solo al entrenar
           layers.RandomTranslation(0.1, 0.1))
variantes = {"sin regularizar": red(), "dropout 0.3": red(drop=0.3),
             "L2 1e-4": red(l2=1e-4),
             "aumento de datos": red(aumento)}
parada = keras.callbacks.EarlyStopping(
    patience=5, restore_best_weights=True)
for nombre, m in variantes.items():
    h = m.fit(x_ent, y_ent, epochs=60, batch_size=128, verbose=0,
              validation_data=(x_val, y_val), callbacks=[parada])
    b = int(np.argmin(h.history["val_loss"]))
    print(f"{nombre:<17} épocas {len(h.history['loss']):2d}  "
          f"entrenamiento {h.history['accuracy'][b]:.3f}  "
          f"validación {h.history['val_accuracy'][b]:.3f}")

m = variantes["dropout 0.3"]                    # la elegida
pred = m.predict(x_pru, verbose=0).argmax(1)
print(f"dropout 0.3 en prueba: {np.mean(pred == y_pru):.4f}")
conf = np.zeros((10, 10), int)
np.add.at(conf, (y_pru, pred), 1)               # matriz de confusión
print("camisa confundida con:", conf[6])
plt.imshow(conf, cmap="Blues")
for i in range(10):
    for j in range(10):
        plt.text(j, i, conf[i, j], ha="center", va="center",
                 fontsize=5, color="w" if conf[i, j] > 500 else "k")
plt.xlabel("clase predicha"); plt.ylabel("clase real")
plt.xticks(range(10)); plt.yticks(range(10))
plt.show()  # plt.savefig("confusion.pdf")

**Prueba.** Sube el dropout a 0.5. ¿Mejora la validación o la red empieza a quedarse corta?

## 2. La convolución a mano (2.7)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x, y), _ = keras.datasets.fashion_mnist.load_data()
print("clase de la imagen:", y[0])        # 9 = botín
img = x[0] / 255.0             # la primera imagen: un botín

def convolucion(img, filtro):
    """Desliza el filtro por la imagen (sin relleno, paso 1)."""
    k = filtro.shape[0]
    alto, ancho = img.shape[0] - k + 1, img.shape[1] - k + 1
    sal = np.zeros((alto, ancho))
    for i in range(alto):
        for j in range(ancho):
            sal[i, j] = np.sum(img[i:i + k, j:j + k] * filtro)
    return sal

def max_pooling(m, t=2):
    a, b = m.shape[0] // t, m.shape[1] // t
    return m[:a * t, :b * t].reshape(a, t, b, t).max(axis=(1, 3))

vertical = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])  # Sobel
horizontal = vertical.T
bv, bh = convolucion(img, vertical), convolucion(img, horizontal)
rp = max_pooling(np.maximum(bh, 0))         # ReLU y reducción
print("imagen", img.shape, "-> convolución", bv.shape,
      "-> pooling", rp.shape)
print(f"respuesta máxima: vertical {np.abs(bv).max():.2f}, "
      f"horizontal {np.abs(bh).max():.2f}")
print("pesos de 32 filtros 3x3:", 32 * (3 * 3 + 1),
      "| pesos de 256 neuronas densas:", 256 * (784 + 1))

fig, ejes = plt.subplots(1, 4, figsize=(4.4, 1.3))
for e, m, t in zip(ejes, [img, bv, bh, rp],
                   ["imagen", "bordes vert.",
                    "bordes horiz.", "ReLU y pooling"]):
    if m is img:
        e.imshow(m, cmap="gray_r")
    else:                                   # 0 en blanco
        e.imshow(m, cmap="RdBu", vmin=-4, vmax=4)
    e.set_title(t, fontsize=6); e.axis("off")
plt.show()  # fig.savefig("convolucion.pdf")

## 3. Una red convolucional (2.7)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x[..., None] / 255.0, x_pru[..., None] / 255.0  # 1 canal

keras.utils.set_random_seed(0)
red = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation="relu"),     # 32 filtros 3x3
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(), layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")])
red.compile("adam", "sparse_categorical_crossentropy",
            metrics=["accuracy"])
red.summary()
parada = keras.callbacks.EarlyStopping(
    patience=3, restore_best_weights=True)
h = red.fit(x[:50_000], y[:50_000], epochs=30, batch_size=128,
            validation_data=(x[50_000:], y[50_000:]),
            callbacks=[parada], verbose=0)
print(f"épocas {len(h.history['loss'])}; prueba "
      f"{red.evaluate(x_pru, y_pru, verbose=0)[1]:.4f}")
pred = red.predict(x_pru, verbose=0).argmax(1)
print(f"camisa: {np.mean(pred[y_pru == 6] == 6):.3f}")

# qué ve la primera capa: 8 de sus 32 mapas para el primer botín
capa1 = keras.Model(red.inputs, red.layers[0].output)
mapas = capa1.predict(x[:1], verbose=0)[0]       # (26, 26, 32)
fig, ejes = plt.subplots(1, 8, figsize=(4.4, 0.75))
for k, e in enumerate(ejes):
    e.imshow(mapas[..., k]); e.axis("off")
plt.show()  # fig.savefig("mapas.pdf")

**Prueba.** Añade a esta red el aumento de datos de la celda 1 (`RandomFlip` y `RandomTranslation`) justo después de `keras.Input`. ¿Ayuda ahora, con una red que sí aprovecha la forma de la imagen?

## 4. Memoria larga: el problema de la suma (2.8)

In [ ]:
import numpy as np
import keras
from keras import layers

def sumas(n, largo, semilla):
    """Problema de la suma: hay que sumar los dos valores marcados,
    uno en la primera mitad de la secuencia y otro en la segunda."""
    rng = np.random.default_rng(semilla)
    v = rng.uniform(0, 1, (n, largo))
    m = np.zeros((n, largo))
    m[np.arange(n), rng.integers(0, largo // 2, n)] = 1
    m[np.arange(n), rng.integers(largo // 2, largo, n)] = 1
    x = np.stack([v, m], axis=-1).astype("float32")
    return x, (v * m).sum(1).astype("float32")

if __name__ == "__main__":          # solo al ejecutar este archivo
    for largo in [10, 50, 100]:
        x, y = sumas(10_000, largo, 0)
        xp, yp = sumas(1000, largo, 1)
        for tipo in [layers.SimpleRNN, layers.LSTM]:
            keras.utils.set_random_seed(0)
            red = keras.Sequential([keras.Input(shape=(largo, 2)),
                                    tipo(32), layers.Dense(1)])
            red.compile("adam", "mse")
            red.fit(x, y, epochs=20, batch_size=64, verbose=0)
            ecm = red.evaluate(xp, yp, verbose=0)
            print(f"{largo:3d} pasos, {tipo.__name__:<9}: error "
                  f"{ecm:.4f}")
    print(f"sin memoria (responder siempre 1): "
          f"{np.mean((yp - 1) ** 2):.4f}")

**Prueba.** Cambia `layers.LSTM` por `layers.GRU`. ¿Resuelve también las secuencias de 100 pasos?

## 5. Imágenes como secuencias y series temporales (2.8)

In [ ]:
import numpy as np
import keras
from keras import layers

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x / 255.0, x_pru / 255.0     # (n, 28, 28): 28 pasos de 28

keras.utils.set_random_seed(0)
red = keras.Sequential([keras.Input(shape=(28, 28)),
                        layers.LSTM(128),        # lee fila a fila
                        layers.Dense(10, "softmax")])
red.compile("adam", "sparse_categorical_crossentropy",
            metrics=["accuracy"])
parada = keras.callbacks.EarlyStopping(
    patience=3, restore_best_weights=True)
h = red.fit(x[:50_000], y[:50_000], epochs=30, batch_size=128,
            validation_data=(x[50_000:], y[50_000:]),
            callbacks=[parada], verbose=0)
print(f"parámetros {red.count_params():,}; "
      f"épocas {len(h.history['loss'])}")
print(f"prueba: {red.evaluate(x_pru, y_pru, verbose=0)[1]:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

rng = np.random.default_rng(0)
t = np.arange(3000)
serie = (np.sin(2 * np.pi * t / 50) + 0.5 * np.sin(2 * np.pi * t / 7)
         + rng.normal(0, 0.2, t.size)).astype("float32")

def ventanas(s, n=30):
    """Cada ejemplo: n valores seguidos; objetivo: el siguiente."""
    x = np.stack([s[i:i + n] for i in range(len(s) - n)])
    return x[..., None], s[n:]

x, y = ventanas(serie[:2500])           # pasado: entrenamiento
xp, yp = ventanas(serie[2500:])          # futuro: prueba
keras.utils.set_random_seed(0)
red = keras.Sequential([keras.Input(shape=(30, 1)),
                        layers.LSTM(32), layers.Dense(1)])
red.compile("adam", "mse")
red.fit(x, y, epochs=20, batch_size=32, verbose=0)
pred = red.predict(xp, verbose=0)[:, 0]
print(f"error medio, repetir el último valor: "
      f"{np.mean(np.abs(xp[:, -1, 0] - yp)):.3f}")
print(f"error medio, LSTM: {np.mean(np.abs(pred - yp)):.3f}")
print(f"mínimo posible, por el ruido: {0.2 * np.sqrt(2 / np.pi):.3f}")

plt.plot(yp[:150], ".", ms=2, label="real")
plt.plot(pred[:150], label="LSTM")
plt.xlabel("paso"); plt.legend(); plt.show()  # plt.savefig("serie.pdf")

## 6. Autocodificador y anomalías (2.9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x.reshape(-1, 784) / 255, x_pru.reshape(-1, 784) / 255
(_, _), (dig, _) = keras.datasets.mnist.load_data()
dig = dig.reshape(-1, 784) / 255              # dígitos: algo distinto

def autocodificador(k):
    keras.utils.set_random_seed(0)
    cod = keras.Sequential([keras.Input(shape=(784,)),
                            layers.Dense(256, "relu"),
                            layers.Dense(k)])     # código: k números
    dec = keras.Sequential([keras.Input(shape=(k,)),
                            layers.Dense(256, "relu"),
                            layers.Dense(784, "sigmoid")])
    auto = keras.Sequential([cod, dec])
    auto.compile("adam", "binary_crossentropy")
    auto.fit(x, x, epochs=20, batch_size=256, verbose=0)  # x -> x
    return cod, auto

def error(auto, datos):                 # error de reconstrucción
    return np.mean((datos - auto.predict(datos, verbose=0)) ** 2, 1)

for k in [2, 32]:
    cod, auto = autocodificador(k)
    e_ropa, e_dig = error(auto, x_pru), error(auto, dig)
    umbral = np.quantile(e_ropa, 0.99)  # 1 % de falsas alarmas
    print(f"código de {k:2d}: error ropa {e_ropa.mean():.4f}, "
          f"dígitos {e_dig.mean():.4f}; dígitos anómalos "
          f"{np.mean(e_dig > umbral):.3f}")
    if k == 2:
        z = cod.predict(x_pru, verbose=0)
        plt.scatter(z[:, 0], z[:, 1], c=y_pru, cmap="tab10", s=0.3)
        plt.colorbar(ticks=range(10)); plt.show()  # plt.savefig("codigos.pdf")

**Ejercicio.** Entrena el autocodificador solo con camisetas y busca, entre todas las prendas de prueba, las 10 con más error de reconstrucción. ¿Qué prendas son?

> LeCun, Y., Bottou, L., Bengio, Y. y Haffner, P. (1998). Gradient-based learning applied to document recognition. *Proceedings of the IEEE*, 86(11), 2278–2324. https://doi.org/10.1109/5.726791
>
> Hochreiter, S. y Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735
>
> Hinton, G. E. y Salakhutdinov, R. R. (2006). Reducing the dimensionality of data with neural networks. *Science*, 313(5786), 504–507. https://doi.org/10.1126/science.1127647